Database Connection

In [1]:
import os

import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.exc import SQLAlchemyError

load_dotenv(find_dotenv())

def connect_to_db():
    host = os.getenv("DB_HOST", "localhost")
    port = os.getenv("DB_PORT", "5432")
    database = os.getenv("DB_NAME")
    user = os.getenv("DB_USER")
    password = os.getenv("DB_PASSWORD")

    if not all([database, user, password]):
        print("Σφάλμα: Δεν βρέθηκαν οι απαραίτητες μεταβλητές στο .env!")
        return None

    try:
        connection_uri = f"postgresql://{user}:{password}@{host}:{port}/{database}"
        engine = create_engine(connection_uri)
        with engine.connect() as connection:
            print(f"Επιτυχής σύνδεση στη βάση '{database}' στο host '{host}'!")
        return engine

    except SQLAlchemyError as e:
        print(f"Σφάλμα κατά τη σύνδεση στη ΒΔ: {e}")
        return None


print("Εκκίνηση σύνδεσης με τη βάση...")
engine = connect_to_db()

if engine is None:
    print("Τερματισμός προγράμματος λόγω αποτυχίας σύνδεσης.")
    exit(1)

Εκκίνηση σύνδεσης με τη βάση...
Επιτυχής σύνδεση στη βάση 'thesis_db' στο host 'dell-micro'!


Import data to dataframe

In [2]:
import pandas as pd

query = "SELECT match_id, x_loc, y_loc, distance_to_goal, angle_to_goal, is_goal FROM understat_shots_normalized"
us_shots = pd.read_sql(query, con=engine)


query = "SELECT match_id, x_loc, y_loc, distance_to_goal, angle_to_goal, is_goal FROM statsbomb_shots_normalized"
sb_shots = pd.read_sql(query, con=engine)

Remove overlapping matches

In [3]:
query = "SELECT understat_match_id FROM mapping_overlapping_matches"
overlapping_matches = pd.read_sql_query(query, con=engine)
overlapping_matches.head()

,understat_match_id
0,923
1,662
2,727
3,890
4,644


In [4]:
print(f"Αρχικές εγγραφές us_shots: {len(us_shots)}")
overlapping_matches_list = overlapping_matches['understat_match_id'].unique()

# Αφαίρεση overlapping matches από το us_shots
us_shots = us_shots[~us_shots['match_id'].isin(overlapping_matches_list)].reset_index(drop=True)

print(f"Overlapping matches: {len(overlapping_matches_list)}")
print(f"Εγγραφές μετά την αφαίρεση: {len(us_shots)}")

Αρχικές εγγραφές us_shots: 605446
Overlapping matches: 1576
Εγγραφές μετά την αφαίρεση: 566725


Drop 'match_id' from DataFrames

In [5]:
us_shots.drop(['match_id'], axis=1, inplace=True)
sb_shots.drop(['match_id'], axis=1, inplace=True)

print(f"{list(us_shots)} και {list(sb_shots)}")

['x_loc', 'y_loc', 'distance_to_goal', 'angle_to_goal', 'is_goal'] και ['x_loc', 'y_loc', 'distance_to_goal', 'angle_to_goal', 'is_goal']


Merge 2 dataframes

In [6]:
shots = pd.concat([us_shots, sb_shots], ignore_index=True)
print(f"Μέγεθος Understat: {len(us_shots)}")
print(f"Μέγεθος StatsBomb: {len(sb_shots)}")
print(f"Μέγεθος Συνολικό : {len(shots)}")

Μέγεθος Understat: 566725
Μέγεθος StatsBomb: 55857
Μέγεθος Συνολικό : 622582


In [7]:
shots.dtypes

x_loc               float64
y_loc               float64
distance_to_goal    float64
angle_to_goal       float64
is_goal               int64
dtype: object

In [7]:
from sklearn.metrics import log_loss, roc_auc_score, brier_score_loss
import numpy as np

all_results = []

def evaluate_model(y_true, y_pred, model_name="Model"):
    loss = log_loss(y_true, y_pred)
    auc = roc_auc_score(y_true, y_pred)
    brier = brier_score_loss(y_true, y_pred)
    actual_goals = int(np.sum(y_true))
    expected_goals = float(np.sum(y_pred))

    print(f"╔{'═'*45}╗")
    print(f"  Αποτελέσματα για: {model_name}")
    print(f"╠{'═'*45}╣")
    print(f"  Log Loss:        {loss:.4f}")
    print(f"  ROC AUC:         {auc:.4f}")
    print(f"  Brier Score:     {brier:.4f}")
    print(f"  Actual Goals:    {actual_goals}")
    print(f"  Expected Goals:  {expected_goals:.1f}")
    print(f"╚{'═'*45}╝\n")

    return {
        "Model": model_name,
        "Log Loss": round(loss, 4),
        "ROC AUC": round(auc, 4),
        "Brier Score": round(brier, 4),
        "Actual Goals": actual_goals,
        "Expected Goals": round(expected_goals, 1)
    }

In [8]:
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_predict, StratifiedKFold

X = shots.drop(columns=['is_goal'])
y = shots['is_goal']

skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
skf_tune = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

BL4.1.1.1 — Default Params + Imbalance: None + Calibration: None

In [9]:
bl4_1_1_1 = LGBMClassifier(
    n_jobs=-1,
    verbose=-1,
    random_state=42
)

predictions_bl4_1_1_1 = cross_val_predict(
    bl4_1_1_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_1_1_1, model_name="BL4.1.1.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.1.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.2792
  ROC AUC:         0.7467
  Brier Score:     0.0788
  Actual Goals:    60584
  Expected Goals:  60584.3
╚═════════════════════════════════════════════╝



BL4.1.1.2 — Default Params + Imbalance: None + Calibration: Platt Scaling

In [10]:
from sklearn.linear_model import LogisticRegression

P_raw = predictions_bl4_1_1_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)

predictions_bl4_1_1_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_1_1_2, model_name="BL4.1.1.2")
all_results.append(res)

C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjec

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.1.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2845
  ROC AUC:         0.7467
  Brier Score:     0.0797
  Actual Goals:    60584
  Expected Goals:  60579.6
╚═════════════════════════════════════════════╝



BL4.1.1.3 — Default Params + Imbalance: None + Calibration: Isotonic Regression

In [11]:
from sklearn.isotonic import IsotonicRegression

iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl4_1_1_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl4_1_1_3, model_name="BL4.1.1.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.1.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2793
  ROC AUC:         0.7462
  Brier Score:     0.0788
  Actual Goals:    60584
  Expected Goals:  60586.8
╚═════════════════════════════════════════════╝



BL4.1.2.1 — Default Params + Imbalance: Class Weight + Calibration: None

In [12]:
bl4_1_2_1 = LGBMClassifier(
    class_weight='balanced',
    n_jobs=-1,
    random_state=42
)

predictions_bl4_1_2_1 = cross_val_predict(
    bl4_1_2_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_1_2_1, model_name="BL4.1.2.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.2.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5853
  ROC AUC:         0.7466
  Brier Score:     0.2012
  Actual Goals:    60584
  Expected Goals:  262783.2
╚═════════════════════════════════════════════╝



BL4.1.2.2 — Default Params + Imbalance: Class Weight + Calibration: Platt Scaling

In [13]:
P_raw = predictions_bl4_1_2_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)

predictions_bl4_1_2_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_1_2_2, model_name="BL4.1.2.2")
all_results.append(res)

C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjec

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.2.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2803
  ROC AUC:         0.7466
  Brier Score:     0.0792
  Actual Goals:    60584
  Expected Goals:  60546.7
╚═════════════════════════════════════════════╝



BL4.1.2.3 — Default Params + Imbalance: Class Weight + Calibration: Isotonic Regression

In [14]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl4_1_2_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl4_1_2_3, model_name="BL4.1.2.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.2.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2794
  ROC AUC:         0.7461
  Brier Score:     0.0788
  Actual Goals:    60584
  Expected Goals:  60585.2
╚═════════════════════════════════════════════╝



BL4.1.3.1 — Default Params + Imbalance: Undersampling + Calibration: None

In [15]:
from imblearn.pipeline import make_pipeline as make_imb_pipeline
from imblearn.under_sampling import RandomUnderSampler

bl4_1_3_1 = make_imb_pipeline(
    RandomUnderSampler(random_state=42),
    LGBMClassifier(
        n_jobs=-1,
        random_state=42
    )
)

predictions_bl4_1_3_1 = cross_val_predict(
    bl4_1_3_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_1_3_1, model_name="BL4.1.3.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.3.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5883
  ROC AUC:         0.7470
  Brier Score:     0.2022
  Actual Goals:    60584
  Expected Goals:  263934.8
╚═════════════════════════════════════════════╝



BL4.1.3.2 — Default Params + Imbalance: Undersampling + Calibration: Platt Scaling

In [16]:
P_raw = predictions_bl4_1_3_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)

predictions_bl4_1_3_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_1_3_2, model_name="BL4.1.3.2")
all_results.append(res)

C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjec

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.3.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2801
  ROC AUC:         0.7470
  Brier Score:     0.0791
  Actual Goals:    60584
  Expected Goals:  60547.7
╚═════════════════════════════════════════════╝



BL4.1.3.3 — Default Params + Imbalance: Undersampling + Calibration: Isotonic Regression

In [17]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl4_1_3_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl4_1_3_3, model_name="BL4.1.3.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.3.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2796
  ROC AUC:         0.7465
  Brier Score:     0.0788
  Actual Goals:    60584
  Expected Goals:  60583.1
╚═════════════════════════════════════════════╝



BL4.1.4.1 — Default Params + Imbalance: Oversampling + Calibration: None

In [18]:
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import BorderlineSMOTE

bl4_1_4_1 = make_imb_pipeline(
    StandardScaler(),
    BorderlineSMOTE(random_state=42),
    LGBMClassifier(
        n_jobs=-1,
        random_state=42
    )
)

predictions_bl4_1_4_1 = cross_val_predict(
    bl4_1_4_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_1_4_1, model_name="BL4.1.4.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.4.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5791
  ROC AUC:         0.7433
  Brier Score:     0.2007
  Actual Goals:    60584
  Expected Goals:  257185.6
╚═════════════════════════════════════════════╝



BL4.1.4.2 — Default Params + Imbalance: Oversampling + Calibration: Platt Scaling

In [19]:
P_raw = predictions_bl4_1_4_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)

predictions_bl4_1_4_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_1_4_2, model_name="BL4.1.4.2")
all_results.append(res)

C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjec

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.4.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2829
  ROC AUC:         0.7433
  Brier Score:     0.0802
  Actual Goals:    60584
  Expected Goals:  60589.9
╚═════════════════════════════════════════════╝



BL4.1.4.3 — Default Params + Imbalance: Oversampling + Calibration: Isotonic Regression

In [20]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl4_1_4_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl4_1_4_3, model_name="BL4.1.4.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.1.4.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2812
  ROC AUC:         0.7427
  Brier Score:     0.0795
  Actual Goals:    60584
  Expected Goals:  60587.1
╚═════════════════════════════════════════════╝



BL4.2.1.1 — HyperParams + Imbalance: None + Calibration: None

In [30]:
import optuna
from sklearn.model_selection import cross_val_predict, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_bt(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 400, 1000, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),

        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),

        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        'class_weight': None,
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()

sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

best_model_bl4_2_1_1 = LGBMClassifier(
    **study_bt.best_params,
    n_jobs=-1,
    random_state=42
)

predictions_bl4_2_1_1 = cross_val_predict(
    best_model_bl4_2_1_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_2_1_1, model_name="BL4.2.1.1")
all_results.append(res)

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 65. Best value: 0.278969: 100%|██████████| 100/100 [52:18<00:00, 31.38s/it]



Βέλτιστες Παράμετροι: {'n_estimators': 950, 'learning_rate': 0.020385880176380506, 'num_leaves': 121, 'max_depth': 5, 'min_child_samples': 98, 'subsample': 0.9089200877768676, 'colsample_bytree': 0.7790078293743563, 'reg_alpha': 9.563226582886122, 'reg_lambda': 0.16745358598796764}
Καλύτερο CV Log Loss: 0.27897
╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.1.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.2789
  ROC AUC:         0.7479
  Brier Score:     0.0787
  Actual Goals:    60584
  Expected Goals:  60582.8
╚═════════════════════════════════════════════╝



In [ ]:
best_params = {
    'n_estimators': 950,
    'learning_rate': 0.020385880176380506,
    'num_leaves': 121,
    'max_depth': 5,
    'min_child_samples': 98,
    'subsample': 0.9089200877768676,
    'colsample_bytree': 0.7790078293743563,
    'reg_alpha': 9.563226582886122,
    'reg_lambda': 0.16745358598796764
}

In [22]:
import optuna
from sklearn.model_selection import cross_val_predict, cross_val_score

optuna.logging.set_verbosity(optuna.logging.WARNING)

def objective_bt(trial):
    params = {
        # Boosting Parameters
        'n_estimators': trial.suggest_int('n_estimators', 600, 1500, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.8, log=True),

        # Tree Structure
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 800, step=50),

        # Subsampling
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

        # Regularization
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 50.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        'class_weight': None,
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()

sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")



Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 82. Best value: 0.278906: 100%|██████████| 100/100 [1:11:09<00:00, 42.70s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 1300, 'learning_rate': 0.07971315324877495, 'num_leaves': 116, 'max_depth': 3, 'min_child_samples': 150, 'subsample': 0.9536693609901703, 'colsample_bytree': 0.7848103576479776, 'reg_alpha': 13.818059682441294, 'reg_lambda': 2.0410723893181272e-07}
Καλύτερο CV Log Loss: 0.27891


In [ ]:
best_params = {
    'n_estimators': 1300,
    'learning_rate': 0.07971315324877495,
    'num_leaves': 116,
    'max_depth': 3,
    'min_child_samples': 150,
    'subsample': 0.9536693609901703,
    'colsample_bytree': 0.7848103576479776,
    'reg_alpha': 13.818059682441294,
    'reg_lambda': 2.0410723893181272e-07
}

In [23]:
best_model_bl4_2_1_1 = LGBMClassifier(
    **study_bt.best_params,
    class_weight=None,
    subsample_freq=1,
    n_jobs=-1,
    random_state=42
)

predictions_bl4_2_1_1 = cross_val_predict(
    best_model_bl4_2_1_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_2_1_1, model_name="BL4.2.1.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.1.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.2789
  ROC AUC:         0.7481
  Brier Score:     0.0787
  Actual Goals:    60584
  Expected Goals:  60582.4
╚═════════════════════════════════════════════╝



BL4.2.1.2 — HyperParams + Imbalance: None + Calibration: Platt Scaling

In [24]:
P_raw = predictions_bl4_2_1_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)

predictions_bl4_2_1_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_2_1_2, model_name="BL4.2.1.2")
all_results.append(res)

C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjec

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.1.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2842
  ROC AUC:         0.7481
  Brier Score:     0.0796
  Actual Goals:    60584
  Expected Goals:  60582.9
╚═════════════════════════════════════════════╝



BL4.2.1.3 — HyperParams + Imbalance: None + Calibration: Isotonic Regression

In [25]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl4_2_1_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl4_2_1_3, model_name="BL4.2.1.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.1.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2790
  ROC AUC:         0.7476
  Brier Score:     0.0787
  Actual Goals:    60584
  Expected Goals:  60585.8
╚═════════════════════════════════════════════╝



BL4.2.2.1 — HyperParams + Imbalance: Class Weight + Calibration: None

In [29]:
# Χρησιμοποιούμαι το ίδιο search space με το 2ο run του Optuna για την εύρεση βέλτιστων παραμέτρων
def objective_bt(trial):
    params = {
        # Boosting Parameters
        'n_estimators': trial.suggest_int('n_estimators', 600, 1500, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.8, log=True),

        # Tree Structure
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 50, 800, step=50),

        # Subsampling
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),

        # Regularization
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 50.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        'class_weight': 'balanced',
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 93. Best value: 0.552878: 100%|██████████| 100/100 [1:28:39<00:00, 53.19s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 1400, 'learning_rate': 0.15419853111966617, 'num_leaves': 78, 'max_depth': 10, 'min_child_samples': 50, 'subsample': 0.9002334833679076, 'colsample_bytree': 0.9309866374594895, 'reg_alpha': 0.008811497343408057, 'reg_lambda': 0.01323001431235397}
Καλύτερο CV Log Loss: 0.55288


In [ ]:
best_params = {
    'n_estimators': 1400,
    'learning_rate': 0.15419853111966617,
    'num_leaves': 78,
    'max_depth': 10,
    'min_child_samples': 50,
    'subsample': 0.9002334833679076,
    'colsample_bytree': 0.9309866374594895,
    'reg_alpha': 0.008811497343408057,
    'reg_lambda': 0.01323001431235397
}

In [31]:
# Refined search space
def objective_bt(trial):
    params = {
        # Boosting Parameters
        'n_estimators': trial.suggest_int('n_estimators', 1000, 1800, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.8, log=True),

        # Tree Structure
        'num_leaves': trial.suggest_int('num_leaves', 60, 140),
        'max_depth': trial.suggest_int('max_depth', 9, 17),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 80, step=5),

        # Subsampling
        'subsample': trial.suggest_float('subsample', 0.75, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.75, 1.0),

        # Regularization
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),

        'class_weight': 'balanced',
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1
    }

    model = LGBMClassifier(**params)

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 48. Best value: 0.535118: 100%|██████████| 100/100 [1:57:15<00:00, 70.35s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 1500, 'learning_rate': 0.1588874440549529, 'num_leaves': 104, 'max_depth': 14, 'min_child_samples': 10, 'subsample': 0.9850722000669339, 'colsample_bytree': 0.8937684551690644, 'reg_alpha': 0.007070677639405962, 'reg_lambda': 0.0018070529477917784}
Καλύτερο CV Log Loss: 0.53512


In [32]:
best_params = {
    'n_estimators': 1500,
    'learning_rate': 0.1588874440549529,
    'num_leaves': 104,
    'max_depth': 14,
    'min_child_samples': 10,
    'subsample': 0.9850722000669339,
    'colsample_bytree': 0.8937684551690644,
    'reg_alpha': 0.007070677639405962,
    'reg_lambda': 0.0018070529477917784
}

In [33]:
best_model_bl4_2_2_1 = LGBMClassifier(
    **study_bt.best_params,
    class_weight='balanced',
    subsample_freq=1,
    n_jobs=-1,
    random_state=42
)

predictions_bl4_2_2_1 = cross_val_predict(
    best_model_bl4_2_2_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_2_2_1, model_name="BL4.2.2.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.2.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5379
  ROC AUC:         0.7169
  Brier Score:     0.1869
  Actual Goals:    60584
  Expected Goals:  217466.9
╚═════════════════════════════════════════════╝



BL4.2.2.2 — HyperParams + Imbalance: Class Weight + Calibration: Platt Scaling

In [34]:
P_raw = predictions_bl4_2_2_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)

predictions_bl4_2_2_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_2_2_2, model_name="BL4.2.2.2")
all_results.append(res)

C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjec

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.2.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2913
  ROC AUC:         0.7169
  Brier Score:     0.0818
  Actual Goals:    60584
  Expected Goals:  60601.5
╚═════════════════════════════════════════════╝



BL4.2.2.3 — HyperParams + Imbalance: Class Weight + Calibration: Isotonic Regression

In [35]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl4_2_2_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl4_2_2_3, model_name="BL4.2.2.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.2.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2873
  ROC AUC:         0.7164
  Brier Score:     0.0804
  Actual Goals:    60584
  Expected Goals:  60583.1
╚═════════════════════════════════════════════╝



BL4.2.3.1 — HyperParams + Imbalance: Undersampling + Calibration: None

In [40]:
from imblearn.under_sampling import RandomUnderSampler


# Γενικό search space
def objective_bt(trial):
    params = {
        # Boosting Parameters
        'n_estimators': trial.suggest_int('n_estimators', 200, 900, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),

        # Tree Structure
        'num_leaves': trial.suggest_int('num_leaves', 20, 90),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100, step=10),

        # Subsampling
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),

        # Regularization
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        # Θα γίνει balanced λόγω του undersampling
        'class_weight': None,
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1
    }

    # Δημιουργία Pipeline: Πρώτα Undersampling, μετά LightGBM
    model = make_imb_pipeline(
        RandomUnderSampler(random_state=42),
        LGBMClassifier(**params)
    )

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 61. Best value: 0.587041: 100%|██████████| 100/100 [18:01<00:00, 10.81s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 900, 'learning_rate': 0.02670946995727843, 'num_leaves': 22, 'max_depth': 7, 'min_child_samples': 100, 'subsample': 0.9080883700962991, 'colsample_bytree': 0.8411782184873076, 'reg_alpha': 3.683552435571946, 'reg_lambda': 0.0010842719066974323}
Καλύτερο CV Log Loss: 0.58704


In [ ]:
best_params = {
    'n_estimators': 900,
    'learning_rate': 0.02670946995727843,
    'num_leaves': 22,
    'max_depth': 7,
    'min_child_samples': 100,
    'subsample': 0.9080883700962991,
    'colsample_bytree': 0.8411782184873076,
    'reg_alpha': 3.683552435571946,
    'reg_lambda': 0.0010842719066974323
}

In [52]:
# Refined search space
def objective_bt(trial):
    params = {
        # Boosting Parameters
        'n_estimators': trial.suggest_int('n_estimators', 800, 1600, step=50),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),

        # Tree Structure
        'num_leaves': trial.suggest_int('num_leaves', 10, 60),
        'max_depth': trial.suggest_int('max_depth', 4, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 80, 360, step=20),

        # Subsampling
        'subsample': trial.suggest_float('subsample', 0.8, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),

        # Regularization
        'reg_alpha': trial.suggest_float('reg_alpha', 0.5, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),

        # Θα γίνει balanced λόγω του undersampling
        'class_weight': None,
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1
    }

    # Δημιουργία Pipeline: Πρώτα Undersampling, μετά LightGBM
    model = make_imb_pipeline(
        RandomUnderSampler(random_state=42),
        LGBMClassifier(**params)
    )

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 23. Best value: 0.586669: 100%|██████████| 100/100 [24:02<00:00, 14.43s/it]


Βέλτιστες Παράμετροι: {'n_estimators': 1000, 'learning_rate': 0.05585828523834694, 'num_leaves': 10, 'max_depth': 10, 'min_child_samples': 140, 'subsample': 0.9991679373737163, 'colsample_bytree': 0.9718657443714958, 'reg_alpha': 6.249802228937746, 'reg_lambda': 0.005082962821793838}
Καλύτερο CV Log Loss: 0.58667


In [56]:
best_model_bl4_2_3_1 = make_imb_pipeline(
    RandomUnderSampler(random_state=42),
    LGBMClassifier(
        **study_bt.best_params,
        subsample_freq=1,
        class_weight=None,
        n_jobs=-1,
        random_state=42,
        verbose=-1
    )
)

predictions_bl4_2_3_1 = cross_val_predict(
    best_model_bl4_2_3_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]


res = evaluate_model(y, predictions_bl4_2_3_1, model_name="BL4.2.3.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.3.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5869
  ROC AUC:         0.7488
  Brier Score:     0.2015
  Actual Goals:    60584
  Expected Goals:  263229.3
╚═════════════════════════════════════════════╝



BL4.2.3.2 — HyperParams + Imbalance: Undersampling + Calibration: Platt Scaling

In [57]:
P_raw = predictions_bl4_2_3_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)

predictions_bl4_2_3_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_2_3_2, model_name="BL4.2.3.2")
all_results.append(res)

C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjec

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.3.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2796
  ROC AUC:         0.7488
  Brier Score:     0.0790
  Actual Goals:    60584
  Expected Goals:  60595.6
╚═════════════════════════════════════════════╝



BL4.2.3.3 — HyperParams + Imbalance: Undersampling + Calibration: Isotonic Regression

In [58]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl4_2_3_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl4_2_3_3, model_name="BL4.2.3.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.3.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2788
  ROC AUC:         0.7483
  Brier Score:     0.0786
  Actual Goals:    60584
  Expected Goals:  60584.0
╚═════════════════════════════════════════════╝



BL4.2.4.1 — HyperParams + Imbalance: Oversampling + Calibration: None

In [50]:
from imblearn.over_sampling import BorderlineSMOTE

def objective_bt(trial):
    params = {
        # Boosting Parameters
        'n_estimators': trial.suggest_int('n_estimators', 600, 1400, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),

        # Tree Structure
        'num_leaves': trial.suggest_int('num_leaves', 30, 120),
        'max_depth': trial.suggest_int('max_depth', 5, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 40, 300, step=20),

        # Subsampling
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),

        # Regularization
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 20.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 20.0, log=True),

        # Θα γίνει balanced λόγω του oversampling
        'class_weight': None,
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1
    }

    # Δημιουργία Pipeline: Πρώτα Oversampling, μετά LightGBM
    model = make_imb_pipeline(
        BorderlineSMOTE(random_state=42),
        LGBMClassifier(**params)
    )

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 94. Best value: 0.520588: 100%|██████████| 100/100 [2:48:30<00:00, 101.10s/it] 


Βέλτιστες Παράμετροι: {'n_estimators': 1300, 'learning_rate': 0.14269684821269685, 'num_leaves': 114, 'max_depth': 10, 'min_child_samples': 40, 'subsample': 0.9908370957275683, 'colsample_bytree': 0.9479147343326063, 'reg_alpha': 1.5848570609573776e-08, 'reg_lambda': 0.0023340302793028334}
Καλύτερο CV Log Loss: 0.52059


In [ ]:
best_params = {
    'n_estimators': 1300,
    'learning_rate': 0.14269684821269685,
    'num_leaves': 114,
    'max_depth': 10,
    'min_child_samples': 40,
    'subsample': 0.9908370957275683,
    'colsample_bytree': 0.9479147343326063,
    'reg_alpha': 1.5848570609573776e-08,
    'reg_lambda': 0.0023340302793028334
}

In [59]:
# Refined search space

def objective_bt(trial):
    params = {
        # Boosting Parameters
        'n_estimators': trial.suggest_int('n_estimators', 1100, 1800, step=100),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),

        # Tree Structure
        'num_leaves': trial.suggest_int('num_leaves', 80, 200),
        'max_depth': trial.suggest_int('max_depth', 8, 15),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 70, step=5),

        # Subsampling
        'subsample': trial.suggest_float('subsample', 0.85, 1.0),
        'subsample_freq': 1,
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.80, 1.0),

        # Regularization
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 5.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),

        # Θα γίνει balanced λόγω του oversampling
        'class_weight': None,
        'n_jobs': -1,
        'random_state': 42,
        'verbose': -1
    }

    # Δημιουργία Pipeline: Πρώτα Oversampling, μετά LightGBM
    model = make_imb_pipeline(
        BorderlineSMOTE(random_state=42),
        LGBMClassifier(**params)
    )

    scores = cross_val_score(model, X, y, cv=skf_tune, scoring='neg_log_loss')
    return -scores.mean()


sampler = optuna.samplers.TPESampler(seed=42)
study_bt = optuna.create_study(direction='minimize', sampler=sampler)

print("Έναρξη Optuna Tuning για Gradient Boosting Trees...")
study_bt.optimize(objective_bt, n_trials=100, show_progress_bar=True)

print("\nΒέλτιστες Παράμετροι:", study_bt.best_params)
print(f"Καλύτερο CV Log Loss: {study_bt.best_value:.5f}")

Έναρξη Optuna Tuning για Gradient Boosting Trees...


Best trial: 95. Best value: 0.512745: 100%|██████████| 100/100 [3:10:05<00:00, 114.06s/it] 


Βέλτιστες Παράμετροι: {'n_estimators': 1600, 'learning_rate': 0.09313204944494176, 'num_leaves': 185, 'max_depth': 15, 'min_child_samples': 10, 'subsample': 0.8870036656465125, 'colsample_bytree': 0.850553329380058, 'reg_alpha': 2.6136452243200404e-06, 'reg_lambda': 1.7130558082232305e-08}
Καλύτερο CV Log Loss: 0.51275


In [ ]:
best_params = {
    'n_estimators': 1600,
    'learning_rate': 0.09313204944494176,
    'num_leaves': 185,
    'max_depth': 15,
    'min_child_samples': 10,
    'subsample': 0.8870036656465125,
    'colsample_bytree': 0.850553329380058,
    'reg_alpha': 2.6136452243200404e-06,
    'reg_lambda': 1.7130558082232305e-08
}

In [61]:
bl4_2_4_1 = make_imb_pipeline(
    StandardScaler(),
    BorderlineSMOTE(random_state=42),
    LGBMClassifier(
        **study_bt.best_params,
        subsample_freq=1,
        class_weight=None,
        n_jobs=-1,
        random_state=42
    )
)

predictions_bl4_2_4_1 = cross_val_predict(
    bl4_2_4_1,
    X,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_2_4_1, model_name="BL4.2.4.1")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.4.1
╠═════════════════════════════════════════════╣
  Log Loss:        0.5128
  ROC AUC:         0.6974
  Brier Score:     0.1724
  Actual Goals:    60584
  Expected Goals:  201801.6
╚═════════════════════════════════════════════╝



BL4.2.4.2 — HyperParams + Imbalance: Oversampling + Calibration: Platt Scaling

In [62]:
P_raw = predictions_bl4_2_4_1.reshape(-1, 1)

platt_calibrator = LogisticRegression(penalty=None, solver='lbfgs', random_state=42)

predictions_bl4_2_4_2 = cross_val_predict(
    platt_calibrator,
    P_raw,
    y,
    cv=skf,
    method='predict_proba'
)[:, 1]

res = evaluate_model(y, predictions_bl4_2_4_2, model_name="BL4.2.4.2")
all_results.append(res)

C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjects\thesis\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Zoe\PycharmProjec

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.4.2
╠═════════════════════════════════════════════╣
  Log Loss:        0.2969
  ROC AUC:         0.6974
  Brier Score:     0.0833
  Actual Goals:    60584
  Expected Goals:  60567.8
╚═════════════════════════════════════════════╝



BL4.2.4.3 — HyperParams + Imbalance: Oversampling + Calibration: Isotonic Regression

In [63]:
iso_calibrator = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds='clip')

predictions_bl4_2_4_3 = cross_val_predict(
    iso_calibrator,
    P_raw,
    y,
    cv=skf
)

res = evaluate_model(y, predictions_bl4_2_4_3, model_name="BL4.2.4.3")
all_results.append(res)

╔═════════════════════════════════════════════╗
  Αποτελέσματα για: BL4.2.4.3
╠═════════════════════════════════════════════╣
  Log Loss:        0.2957
  ROC AUC:         0.6968
  Brier Score:     0.0830
  Actual Goals:    60584
  Expected Goals:  60580.8
╚═════════════════════════════════════════════╝



Export Results to csv

In [65]:
df_results = pd.DataFrame(all_results)

for col in ['model_name', 'Model', 'model']:
    if col in df_results.columns:
        df_results = df_results.drop_duplicates(subset=[col], keep='last')
        break

df_results.to_csv('BL4-results.csv', index=False, encoding='utf-8-sig')
display(df_results)

,Model,Log Loss,ROC AUC,Brier Score,Actual Goals,Expected Goals
0,BL4.1.1.1,0.2792,0.7467,0.0788,60584,60584.3
1,BL4.1.1.2,0.2845,0.7467,0.0797,60584,60579.6
2,BL4.1.1.3,0.2793,0.7462,0.0788,60584,60586.8
3,BL4.1.2.1,0.5853,0.7466,0.2012,60584,262783.2
4,BL4.1.2.2,0.2803,0.7466,0.0792,60584,60546.7
5,BL4.1.2.3,0.2794,0.7461,0.0788,60584,60585.2
6,BL4.1.3.1,0.5883,0.7470,0.2022,60584,263934.8
7,BL4.1.3.2,0.2801,0.7470,0.0791,60584,60547.7
8,BL4.1.3.3,0.2796,0.7465,0.0788,60584,60583.1
9,BL4.1.4.1,0.5791,0.7433,0.2007,60584,257185.6
